## Gold Layer: RFM 客户分群 — PySpark DataFrame API 版

**职责**: 从 Silver 层读取订单数据，用 PySpark DataFrame API 实现 RFM 打分和客户分群。

**为什么又做一遍 RFM？**
- dbt 中 `dim_customers` 用 SQL 实现了完整 RFM（ntile 窗口函数 + 中文标签）
- 本 notebook 用 PySpark DataFrame API 实现同样的逻辑——展示跨技术栈能力
- 面试叙事：**同一个 RFM 模型，SQL (dbt) 和 PySpark DataFrame API 两种实现，适用于不同数据规模场景**

**API 知识点覆盖**:
- `Window.orderBy()` + `rowsBetween()` — 窗口定义
- `F.ntile()` — 五分位打分
- `F.when()` — 条件赋值（PySpark 版 CASE WHEN）
- `F.agg()` + `F.countDistinct()` / `F.sum()` — 聚合
- `F.datediff()` / `F.max()` — 日期函数

**输入**: `silver_fact_orders`
**输出**: `gold_rfm_customer_segments` Delta 表

In [0]:
# ============================================================
# 导入 PySpark 函数库
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# ============================================================
# 从 Silver 层读取数据
# ============================================================

# 读取 silver_fact_orders，取 RFM 计算所需的列
df = spark.table("silver_fact_orders").select(
    "customer_unique_id",
    "order_id",
    "order_purchase_timestamp",
    "price"
).filter(
    F.col("order_purchase_timestamp").isNotNull() &
    F.col("price").isNotNull()
)

print(f"📥 读取 Silver 数据，共 {df.count():,} 行")

In [0]:
# ============================================================
# 计算 RFM 指标（按 customer_unique_id 聚合）
# - Recency: 最近购买日期距数据集中最后一天的间隔
# - Frequency: 购买次数（去重 order_id）
# - Monetary: 累计消费金额
# ============================================================

# 先找到数据集中最后一天（作为 Recency 的参考点）
max_date = df.agg(F.max("order_purchase_timestamp")).collect()[0][0]
print(f"📅 数据集最后日期: {max_date.date()}")

# 按客户聚合计 RFM 基础指标
rfm_base = (
    df
    .groupBy("customer_unique_id")
    .agg(
        F.max("order_purchase_timestamp").alias("last_purchase_date"),
        F.countDistinct("order_id").alias("frequency"),
        F.sum("price").alias("monetary")
    )
    .withColumn(
        "recency",
        F.abs(F.datediff(F.lit(max_date), F.col("last_purchase_date")))
    )
)

In [0]:
# ============================================================
# 🎯 核心：PySpark ntile() 窗口函数 — RFM 五分位打分
# ============================================================

# 定义窗口：对全表进行排序分桶
# recency 越小越好 → 倒序排列（值小的排在桶 5，值大的在桶 1）
# frequency 越大越好 → 正序排列
# monetary 越大越好 → 正序排列
window_all = Window.orderBy(F.col("recency").desc())

rfm_scored = (
    rfm_base
    .withColumn("r_score", F.ntile(5).over(Window.orderBy(F.col("recency").desc())))
    .withColumn("f_score", F.ntile(5).over(Window.orderBy("frequency")))
    .withColumn("m_score", F.ntile(5).over(Window.orderBy("monetary")))
)

In [0]:
# ============================================================
# 计算高低分界线和客户分群标签
# 使用 F.when() + F.otherwise() — PySpark 版 CASE WHEN
# ============================================================

# 计算各分数的平均值（用作文本标签的高/低分界线）
avg_scores = rfm_scored.agg(
    F.avg("r_score").alias("r_avg"),
    F.avg("f_score").alias("f_avg"),
    F.avg("m_score").alias("m_avg")
).collect()[0]

# 用 F.when() 实现条件赋值（对应 SQL 的 CASE WHEN）
gold_rfm = rfm_scored.withColumn(
    "r_level",
    F.when(F.col("r_score") >= F.lit(avg_scores.r_avg), "高").otherwise("低")
).withColumn(
    "f_level",
    F.when(F.col("f_score") >= F.lit(avg_scores.f_avg), "高").otherwise("低")
).withColumn(
    "m_level",
    F.when(F.col("m_score") >= F.lit(avg_scores.m_avg), "高").otherwise("低")
).withColumn(
    "customer_segment",
    F.when(
        (F.col("r_level") == "高") & (F.col("f_level") == "高") & (F.col("m_level") == "高"), "重要价值客户"
    ).when(
        (F.col("r_level") == "高") & (F.col("f_level") == "高") & (F.col("m_level") == "低"), "重要保持客户"
    ).when(
        (F.col("r_level") == "高") & (F.col("f_level") == "低") & (F.col("m_level") == "高"), "重要发展客户"
    ).when(
        (F.col("r_level") == "低") & (F.col("f_level") == "高") & (F.col("m_level") == "高"), "重要挽留客户"
    ).otherwise("一般客户")
)

In [0]:
# ============================================================
# 预览分群结果
# ============================================================

print("📊 RFM 分群结果预览（前 10 行）：")
gold_rfm.select(
    "customer_unique_id", "recency", "frequency", "monetary",
    "r_score", "f_score", "m_score",
    "customer_segment"
).show(10, truncate=False)

print("\n📊 各分群客户数：")
gold_rfm.groupBy("customer_segment").count().orderBy("count", ascending=False).show()

In [0]:
# ============================================================
# 写入 Gold 层 Delta 表
# ============================================================

gold_rfm.write.format("delta").mode("overwrite").saveAsTable("gold_rfm_customer_segments")

row_count = gold_rfm.count()
print(f"✅ gold_rfm_customer_segments 写入完成 — {row_count:,} 位客户")

---

### 与 dbt dim_customers 的对照

| 维度 | dbt (SQL) | Spark (PySpark API) |
|------|-----------|---------------------|
| 窗口函数 | `ntile(5) over(order by ...)` | `F.ntile(5).over(Window.orderBy(...))` |
| 条件赋值 | `CASE WHEN ... THEN ... ELSE ... END` | `F.when(...).when(...).otherwise(...)` |
| 增量处理 | `is_incremental()` + merge | mode("overwrite") 全量覆盖 |
| 适用场景 | 每日调度，对接 BI | 一次性特征工程，离线批量计算 |

### 下一步
Spark 实验完成。生产管道转到 dbt + Airflow + BI。